<a href="https://colab.research.google.com/github/fayzi-dev/ObjectTracking_YOLO8/blob/main/ObjectTracking_YOLO8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Installing libraries


In [2]:
!pip install ultralytics opencv-python numpy

# Import libraries

In [28]:
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import Video


# بارگذاری مدل YOLOv8 (کوچک‌ترین نسخه)
model = YOLO("yolov8n.pt")

# خواندن ویدیو
cap = cv2.VideoCapture("/content/sample_video.mp4")

# تنظیم ذخیره خروجی ویدیو
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # کدک
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("output.mp4", fourcc, fps, (w, h))

# مقداردهی اولیه برای ردیابی
prev_boxes = []
prev_ids = []
next_id = 0

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area if union_area > 0 else 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # تشخیص اشیاء
    results = model(frame)[0]
    curr_boxes = []
    curr_ids = []

    for box in results.boxes.xyxy:
        b = box.cpu().numpy()
        curr_boxes.append(b)

    # ردیابی با IoU ساده
    for box in curr_boxes:
        best_iou = 0
        best_idx = -1
        for i, prev_box in enumerate(prev_boxes):
            iou_val = iou(box, prev_box)
            if iou_val > best_iou:
                best_iou = iou_val
                best_idx = i

        if best_iou > 0.5:
            curr_ids.append(prev_ids[best_idx])
        else:
            curr_ids.append(next_id)
            next_id += 1

    # رسم جعبه‌ها و ID
    for i, box in enumerate(curr_boxes):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID {curr_ids[i]}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # ذخیره فریم در ویدیو خروجی
    out.write(frame)

    # نمایش هم‌زمان
    Video("output.mp4", embed=True)

    # به‌روزرسانی جعبه‌ها
    prev_boxes = curr_boxes
    prev_ids = curr_ids

cap.release()
out.release()
cv2.destroyAllWindows()

from ultralytics import YOLO


0: 384x640 17 persons, 20.4ms
Speed: 3.4ms preprocess, 20.4ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 1 tie, 1 cell phone, 12.9ms
Speed: 3.4ms preprocess, 12.9ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 persons, 1 cell phone, 24.4ms
Speed: 11.1ms preprocess, 24.4ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 1 cell phone, 11.4ms
Speed: 3.3ms preprocess, 11.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 16 persons, 1 cell phone, 9.4ms
Speed: 3.0ms preprocess, 9.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 17 persons, 8.7ms
Speed: 3.2ms preprocess, 8.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 persons, 9.5ms
Speed: 3.3ms preprocess, 9.5ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 14 persons, 1 cell phone, 9.2ms
Sp